# 层与块

## 层（Layer）
层是神经网络中最基础的运算单元。它接收前一层的输入，通过特定的数学变换（如矩阵乘法、卷积或非线性激活函数）后输出给下一层。

核心作用：负责单一维度的计算或特征变换。

常见类型：

- 全连接层（Dense/Linear）：将输入的每一个节点与输出的每一个节点相连，常用于特征组合。
- 卷积层（Convolutional Layer, Conv）：通过滑动局部窗口提取图像或序列的局部空间特征。
- 激活层（Activation Layer）：如 ReLU、Sigmoid，引入非线性能力，使网络能拟合复杂函数。
- 池化层（Pooling Layer）：降低特征图维度，减少参数量并保持平移不变性。
- 归一化层（Normalization Layer）：如 LayerNorm、BatchNorm，用于加速训练并稳定梯度。

## 块（Block）

随着深度学习模型越变越深，直接按“层”逐个堆叠会导致代码冗余且难以维护。块是由多个“层”组合而成的可重复逻辑单元，通常封装了某种特定设计模式或算法思想。

核心作用：提高代码复用性，实现模块化设计，解决深层网络训练中的难题（如梯度消失）。

经典示例：

- 残差块（Residual Block / ResNet Block）：包含卷积层、归一化层和激活层，并通过“跨层连接（Skip Connection）”将输入直接加到输出上，解决深层网络训练退化问题- - 
- Transformer 块（Transformer Block）：结合了多头自注意力机制（Multi-Head Attention）、Feed-Forward 网络以及层归一化（LayerNorm），是现代大语言模型和 Vision Transformer 的核心单元。
- Inception 块：在同一个块内并行运行不同尺寸的卷积核，将提取的不同尺度特征拼接在一起。

## 自定义块

In [7]:
import torch 
from torch import nn
from torch.nn import functional as F  # 这个包下有很多实用的函数

X = torch.rand(2, 20)

In [8]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

In [9]:
net = MLP()
net(X)

tensor([[ 0.1195,  0.0029,  0.1006, -0.0857, -0.0202, -0.1694, -0.1206, -0.3522,
         -0.1735, -0.0759],
        [ 0.1203,  0.0128,  0.1015,  0.0637, -0.0742, -0.0741, -0.0465, -0.2643,
         -0.1353, -0.0020]], grad_fn=<AddmmBackward0>)

同时，我们也可以自定义顺序块

In [10]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X